# Прогноз спроса на билеты ХК «Авангард»

Нужно оценить полное число обычных билетов для 17 матчей сезона 2025/26
в восьми ценовых зонах: **17 × 8 = 136 прогнозов**. Целевая величина
`tickets_total` включает все обычные продажи, в том числе уже состоявшиеся.
Абонементы в неё не входят.

Данные доступны на утро **15 декабря 2025 года**. Продажи известны по
14 декабря включительно. Для шести будущих матчей продажи уже открыты,
для остальных одиннадцати собственных продаж ещё нет.

Прогноз должен быть целым, неотрицательным и находиться между уже проданным
и числом доступных мест: `seats − season_tickets`. Метрика соревнования —
RMSE, корень среднего квадрата ошибки в билетах. Чем он меньше, тем лучше.

## 1. Данные и целевая переменная

Используем четыре файла организаторов. `matches.csv` содержит расписание,
`zones.csv` — ценовые зоны и вместимость, `sales_daily.csv` — обычные продажи
за день, `submission_template.csv` — пары матч–зона в нужном порядке.
Внешних данных нет. Результаты КХЛ и продажи абонементов не входят в признаки.

В обучении 85 завершённых матчей. Их ответы получаем суммированием дневных
продаж каждой пары матч–зона. Отсутствие дневной записи означает ноль продаж.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

matches = pd.read_csv(root / "data/matches.csv", parse_dates=["date", "sales_open"])
zones = pd.read_csv(root / "data/zones.csv")
sales = pd.read_csv(root / "data/sales_daily.csv", parse_dates=["date"])
template = pd.read_csv(root / "data/submission_template.csv")

keys = ["match_id", "zone"]
forecast_date = pd.Timestamp("2025-12-15")

## 2. Признаки из расписания

Зона, соперник, день недели, месяц и время начала известны заранее.
Их используем как категории: например, «понедельник» — отдельная метка,
а не число, которое в семь раз меньше «воскресенья».
Номер домашнего матча в сезоне используем как числовой признак.

Длительность продаж `sales_length` — разность дат матча и открытия продаж,
без добавления единицы. Она понадобится для выбора исторических аналогов.
Окончательные исторические абонементы не используем: неизвестно,
сколько из них было доступно на момент раннего прогноза.

In [2]:
# Доступен на момент прогноза: заранее известное расписание.
calendar = matches.sort_values("date").copy()
calendar["weekday"] = calendar["date"].dt.dayofweek
calendar["month"] = calendar["date"].dt.month
calendar["season_match_number"] = calendar.groupby("season").cumcount() + 1
calendar["sales_length"] = (calendar["date"] - calendar["sales_open"]).dt.days

start_time = pd.to_datetime(calendar["time"], format="%H:%M")
calendar["start_minutes"] = start_time.dt.hour * 60 + start_time.dt.minute

train_matches = calendar.loc[calendar["part"] == "train"]
train = train_matches.merge(zones[keys + ["seats"]], on="match_id")

train_sales = sales.loc[sales["match_id"].isin(train_matches["match_id"])]
totals = train_sales.groupby(keys)["tickets"].sum().rename("tickets_total")
train = train.merge(totals, on=keys, how="left")
train["tickets_total"] = train["tickets_total"].fillna(0).astype(int)

print(f"Обучение: {train['match_id'].nunique()} матчей, {len(train)} пар матч–зона.")

Обучение: 85 матчей, 680 пар матч–зона.


## 3. Уровень спроса каждой зоны

Для начала сравниваем решение с простым ориентиром — средним итогом зоны
по доступным завершённым матчам. Для модели дополнительно учитываем сезон:
спрос одной зоны может заметно измениться между сезонами.

Обозначим общее среднее зоны как $\bar y_z$, среднее зоны в текущем сезоне
как $\bar y_{z,s}$, число доступных матчей этого сезона как $n_{z,s}$.
Сезонный уровень:

$$\mu_{z,s} = (1-w)\bar y_z + w\bar y_{z,s},
\qquad w = \frac{n_{z,s}}{n_{z,s}+8}.$$

Восемь — сила сглаживания: пока матчей нового сезона мало, сохраняем
больше опоры на общую историю. Если нового сезона в истории ещё нет,
используем только общее среднее. В проверке все средние считаются заново
из истории, доступной к конкретному моменту прогноза.

In [3]:
season_prior = 8


def zone_mean(history, query):
    means = history.groupby("zone")["tickets_total"].mean()
    return query["zone"].map(means).to_numpy()


def season_level(history, query):
    # Доступен на момент прогноза: ответы завершённых матчей из history.
    general = zone_mean(history, query)
    statistics = history.groupby(["zone", "season"])["tickets_total"].agg(
        ["mean", "count"]
    )
    query_groups = pd.MultiIndex.from_frame(query[["zone", "season"]])
    statistics = statistics.reindex(query_groups)

    count = statistics["count"].fillna(0).to_numpy()
    weight = count / (count + season_prior)
    current = statistics["mean"].fillna(0).to_numpy()

    return (1 - weight) * general + weight * current

## 4. Поправка календаря и соперника: Ridge

Ridge — линейная регрессия со штрафом за слишком большие коэффициенты.
Это помогает сдерживать случайные эффекты, когда примеров немного.
Категории кодируем отдельными столбцами, номер матча приводим к общей шкале.
Кодировщик, масштабирование и регрессия обучаются только на доступной истории.

Модель учит отклонение от среднего своей зоны и сезона:
$y_i-\bar y_{z,s}$. Это преобразование **обучающих ответов**, а не признак
с неизвестным итогом будущего матча. Прогноз до учёта собственных продаж:
$P_0=\mu_{z,s}+\widehat{\text{отклонение}}$.

Сила штрафа `alpha=10`. Параметры выбраны по временной проверке,
результаты которой приведены ниже.

In [4]:
categorical_features = ["zone", "opponent_khl", "weekday", "month", "start_minutes"]
numeric_features = ["season_match_number"]


def model_features(frame):
    features = frame[categorical_features + numeric_features].copy()
    features[categorical_features] = features[categorical_features].astype(str)
    return features


def calendar_forecast(history, query):
    # Доступен на момент прогноза: расписание и завершённая история.
    preprocessing = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features),
        ("number", StandardScaler(), numeric_features),
    ])
    model = make_pipeline(preprocessing, Ridge(alpha=10, random_state=42))

    centers = history.groupby(["zone", "season"])["tickets_total"].transform("mean")
    deviations = history["tickets_total"] - centers
    model.fit(model_features(history), deviations)

    return season_level(history, query) + model.predict(model_features(query))

## 5. Что известно о собственных продажах

Для матчей с открытыми продажами учитываем отклонение фактических ранних
продаж от ожидаемых. Для сравнения ищем завершённые исторические матчи
на том же числе дней до начала игры.

Обозначения:

| Обозначение | Смысл |
|---|---|
| $D$ | Дата матча |
| $O$ | Дата открытия продаж |
| $C$ | Утро прогноза |
| $L=D-O$ | Длительность окна продаж |
| `sold` | Продажи с открытия до утра $C$, день $C$ не включён |
| $f$ | Ожидаемая доля полного итога на этой стадии |
| $b$ | Связь отклонения ранних продаж с отклонением полного итога |

В аналоги входят только матчи с $L\ge28$, завершённые к утру $C$.
Для аналога с датой $D_a$ продажи обрезаются на $C_a=D_a-(D-C)$.
Если к $C_a$ продажи ещё не открылись, аналог исключается.
Нулевые продажи после открытия остаются в истории.

In [5]:
sales_groups = {key: group for key, group in sales.groupby(keys)}


def sales_before(match_id, zone, cutoff):
    # Доступен на момент cutoff: продажи только за даты строго до него.
    daily = sales_groups.get((match_id, zone))
    if daily is None:
        return 0

    known = daily.loc[daily["date"] < cutoff, "tickets"]
    return known.sum()


def matched_history(history, match_id, cutoff):
    match_date = calendar.set_index("match_id").at[match_id, "date"]
    remaining_days = (match_date - cutoff).days

    analogs = history.loc[
        (history["sales_length"] >= 28) & (history["tickets_total"] > 0)
    ].copy()
    analogs["sales_cutoff"] = analogs["date"] - pd.Timedelta(days=remaining_days)
    analogs = analogs.loc[analogs["sales_cutoff"] > analogs["sales_open"]].copy()

    analogs["early"] = [
        sales_before(row.match_id, row.zone, row.sales_cutoff)
        for row in analogs.itertuples()
    ]
    return analogs

## 6. Как ранние продажи меняют прогноз

В аналогах центрируем ранние продажи $x$ и полный итог $y$ относительно
средних своей зоны и сезона. Учим один общий коэффициент
$b=\sum xy/\sum x^2$, ограниченный от 0 до 3.
При нулевом разбросе ранних продаж принимаем $b=0$.
Неотрицательная связь не позволяет повышенным ранним продажам уменьшать
прогноз; верхнее ограничение сдерживает слишком сильную экстраполяцию.

Для каждой зоны считаем среднюю долю ранних продаж `early / tickets_total`.
Долю текущего сезона сглаживаем к общей истории так же, как уровень спроса,
с силой 8. Получаем $f$: сколько обычно успевают продать на этой стадии.
Ожидаемые ранние продажи для будущего матча равны $fP_0$.

Итоговая поправка:

$$P=P_0+0.5\,b\,(sold-fP_0).$$

Вес 0.5 означает половинное доверие сигналу ранних продаж. Если аналогов
меньше пяти, используем $P_0$. Для матчей без собственных продаж поправки нет.
Эти параметры фиксированы для всей дальнейшей проверки и итогового прогноза.

In [6]:
def sales_correction(analogs, query, initial_prediction):
    if analogs["match_id"].nunique() < 5:
        return np.zeros(len(query))

    groups = ["zone", "season"]
    x = analogs["early"] - analogs.groupby(groups)["early"].transform("mean")
    y = analogs["tickets_total"] - analogs.groupby(groups)["tickets_total"].transform("mean")

    variance = float(x @ x)
    slope = float(np.clip((x @ y) / variance, 0, 3)) if variance > 0 else 0

    analogs = analogs.assign(fraction=analogs["early"] / analogs["tickets_total"])
    overall = query["zone"].map(analogs.groupby("zone")["fraction"].mean()).to_numpy()
    statistics = analogs.groupby(groups)["fraction"].agg(["mean", "count"])
    statistics = statistics.reindex(pd.MultiIndex.from_frame(query[groups]))

    count = statistics["count"].fillna(0).to_numpy()
    weight = count / (count + season_prior)
    fraction = (1 - weight) * overall + weight * statistics["mean"].fillna(0).to_numpy()
    expected_sales = fraction * initial_prediction

    return 0.5 * slope * (query["sold"].to_numpy() - expected_sales)

Для одного момента прогноза сначала обучаем календарную модель на доступной
истории. Затем добавляем поправку отдельно для каждого матча с продажами.
Итоги проверяемых матчей в эту функцию не передаются.

In [7]:
def forecast_tickets(history, query):
    initial = calendar_forecast(history, query)
    prediction = pd.Series(initial, index=query.index)

    with_sales = query.loc[query["has_sales"]]
    for match_id, match_query in with_sales.groupby("match_id"):
        cutoff = match_query["cutoff"].iloc[0]
        analogs = matched_history(history, match_id, cutoff)
        base = prediction.loc[match_query.index].to_numpy()
        correction = sales_correction(analogs, match_query, base)
        prediction.loc[match_query.index] += correction

    return prediction.to_numpy()

## 7. Проверка по времени

Проверяем на трёх последовательных окнах по 17 матчей:
M035–M051, M052–M068 и M069–M085. Все восемь зон матча остаются вместе.
В обучение входят только матчи **до начала окна**, завершённые строго
до календарного дня конкретного прогноза. Проверяемые матчи не пополняют
историю внутри окна, даже если к более поздней дате они уже завершились.

Чтобы воспроизвести условия будущих матчей, используем два сценария:

| Сценарий | Момент прогноза | Какие матчи проверяем |
|---|---|---|
| Без продаж | Утро открытия продаж | Все 17 матчей окна |
| С продажами | Утро после 7 и 27 календарных дней продаж | Одна общая выборка с $L\ge28$ |

Последний доступный день продаж — предыдущий день перед утром прогноза.
Внутри каждой пары матч–зона квадраты ошибок срезов 27/7 дней смешиваем
с весами 2/3 и 1/3: это соответствует четырём и двум будущим матчам
с такими сроками наблюдения. Каждый исторический матч сохраняет одинаковый вес.

Дополнительно показываем объём матчей с длительностью продаж 28–42 дня,
как у будущих матчей. Это диагностическая подвыборка; основной состав проверки остаётся L ≥ 28.

In [8]:
windows = [(35, 51), (52, 68), (69, 85)]
example_parts = []

for first, last in windows:
    first_id, last_id = f"M{first:03}", f"M{last:03}"
    checking = train.loc[train["match_id"].between(first_id, last_id)]

    for days, weight in [(0, 1), (27, 2 / 3), (7, 1 / 3)]:
        sample = checking if days == 0 else checking.loc[checking["sales_length"] >= 28]
        sample = sample.drop(columns="tickets_total").copy()
        sample["window"] = f"{first_id}–{last_id}"
        sample["window_start"] = first_id
        sample["has_sales"] = days > 0
        sample["scenario"] = "С продажами" if days else "Без продаж"
        sample["days_observed"] = days
        sample["weight"] = weight
        sample["cutoff"] = sample["sales_open"] + pd.Timedelta(days=days)
        sample["sold"] = [
            sales_before(row.match_id, row.zone, row.cutoff)
            for row in sample.itertuples()
        ]
        example_parts.append(sample)

examples = pd.concat(example_parts, ignore_index=True)
coverage = examples.groupby(["window", "scenario"])["match_id"].nunique().unstack()
similar_examples = examples.loc[
    examples["has_sales"] & examples["sales_length"].between(28, 42)
]
coverage["С продажами, L 28–42"] = similar_examples.groupby("window")["match_id"].nunique()
display(coverage)

scenario,Без продаж,С продажами,"С продажами, L 28–42"
window,,,
M035–M051,17,10,10
M052–M068,17,17,12
M069–M085,17,9,9


### Общие ограничения прогноза

Оба метода округляем одинаково: ближайшее целое, ровно 0.5 вверх.
Затем применяем минимум уже проданного и максимум вместимости.
В исторической проверке верхняя граница — `seats`: число физических мест
считаем известным заранее. Исторические итоговые `season_tickets`
не подставляем, потому что на раннюю дату их число неизвестно.
Для будущих матчей предоставленная граница — `seats − season_tickets`.

Оставляем две проверки реальных проблем. Если модель вернула NaN или
бесконечность, такого прогноза для сдачи не существует. Если `sold` больше
вместимости, границы противоречат друг другу и корректного ответа дать нельзя.
В этих случаях останавливаем расчёт с понятным сообщением.

In [9]:
def apply_bounds(prediction, sold, capacity):
    prediction = np.asarray(prediction)
    sold = np.asarray(sold)
    capacity = np.asarray(capacity)

    if not np.isfinite(prediction).all():
        raise ValueError("Модель вернула пропуск или бесконечность вместо прогноза.")
    if (sold > capacity).any():
        raise ValueError("Уже продано больше вместимости: необходимо проверить исходные данные.")

    rounded = np.floor(prediction + 0.5)
    return np.clip(rounded, sold, capacity).astype(int)

Для каждого окна и утра прогноза отдельно отбираем завершённую историю
и заново обучаем модель. Прогнозируем полный итог каждого проверяемого матча,
затем сравниваем его с известным итогом. Простое среднее по зоне проходит
ровно те же примеры, даты, округление и ограничения.

In [10]:
methods = {
    "Среднее по зоне": zone_mean,
    "Ridge + ранние продажи": forecast_tickets,
}
actual_totals = train.set_index(keys)["tickets_total"]
result_parts = []

for name, method in methods.items():
    for (window, cutoff), query in examples.groupby(["window", "cutoff"]):
        first_id = query["window_start"].iloc[0]
        history = train.loc[
            (train["match_id"] < first_id) & (train["date"] < cutoff)
        ]
        raw = method(history, query)
        prediction = apply_bounds(raw, query["sold"], query["seats"])

        detail = query[
            keys + ["window", "scenario", "days_observed", "weight", "sales_length"]
        ].copy()
        query_pairs = pd.MultiIndex.from_frame(query[keys])
        detail["tickets_total"] = actual_totals.reindex(query_pairs).to_numpy()
        detail["prediction"] = prediction
        detail["method"] = name
        detail["squared_error"] = (detail["prediction"] - detail["tickets_total"]) ** 2
        result_parts.append(detail)

validation = pd.concat(result_parts, ignore_index=True)

## 8. Результаты проверки

Сначала усредняем квадраты ошибок срезов **внутри пары матч–зона**,
затем между парами. Получаем $MSE_{w,0}$ без продаж и $MSE_{w,1}$ с продажами.
В будущем наборе 11 матчей без продаж и 6 с продажами, поэтому общая ошибка
окна:

$$R_w=\sqrt{\frac{11}{17}MSE_{w,0}+\frac{6}{17}MSE_{w,1}}.$$

Сводный показатель по трём окнам:

$$S=0.25R_1+0.25R_2+0.50R_3.$$

Более близкое по времени окно получает больший вес. $S$ — взвешенная
историческая оценка, а не известный RMSE скрытого набора соревнования.
В таблице показываем обе сценарные ошибки и $R$ для обоих методов.

Последняя колонка показывает ошибку на матчах с L от 28 до 42 дней.
Она описывает переносимость на похожие окна продаж и не входит в расчёт S.

In [11]:
validation["weighted_error"] = validation["weight"] * validation["squared_error"]
pair_errors = validation.groupby(
    ["method", "window", "scenario"] + keys
)["weighted_error"].sum().reset_index()

scenario_mse = pair_errors.groupby(
    ["method", "window", "scenario"]
)["weighted_error"].mean().unstack("scenario")

window_results = np.sqrt(scenario_mse)
window_results["R"] = np.sqrt(
    (11 / 17) * scenario_mse["Без продаж"]
    + (6 / 17) * scenario_mse["С продажами"]
)
similar = validation.loc[
    (validation["scenario"] == "С продажами") & validation["sales_length"].between(28, 42)
]
similar_pairs = similar.groupby(["method", "window"] + keys)["weighted_error"].sum()
window_results["С продажами, L 28–42"] = np.sqrt(
    similar_pairs.groupby(["method", "window"]).mean()
)
display(window_results.round(3))

window_weights = np.array([0.25, 0.25, 0.50])
score_rows = []
for name, group in window_results.groupby("method"):
    group = group.sort_index(level="window")
    score_rows.append({
        "Метод": name,
        "S без продаж": window_weights @ group["Без продаж"].to_numpy(),
        "S с продажами": window_weights @ group["С продажами"].to_numpy(),
        "S": window_weights @ group["R"].to_numpy(),
    })

scores = pd.DataFrame(score_rows).set_index("Метод")
display(scores.round(3))

scenario                          Без продаж  С продажами        R  \
method                 window                                        
Ridge + ранние продажи M035–M051     156.285      136.536  149.613   
                       M052–M068      73.740       66.976   71.426   
                       M069–M085     150.871      127.162  142.952   
Среднее по зоне        M035–M051     155.995      151.394  154.386   
                       M052–M068     105.410      102.931  104.542   
                       M069–M085     157.123      132.333  148.846   

scenario                          С продажами, L 28–42  
method                 window                           
Ridge + ранние продажи M035–M051               136.536  
                       M052–M068                65.291  
                       M069–M085               127.162  
Среднее по зоне        M035–M051               151.394  
                       M052–M068                99.279  
                       M069–M085               132.333

,S без продаж,S с продажами,S
Метод,,,
Ridge + ранние продажи,132.941,114.459,126.736
Среднее по зоне,143.913,129.748,139.155


### Ошибки по ценовым зонам

Для каждой зоны считаем общий RMSE окна с теми же весами двух сценариев.
Это позволяет увидеть, какие зоны дают основную ошибку, без вывода
большой таблицы отдельных матчей.

In [12]:
chosen_errors = pair_errors.loc[pair_errors["method"] == "Ridge + ранние продажи"]
zone_mse = chosen_errors.groupby(
    ["zone", "window", "scenario"]
)["weighted_error"].mean().unstack("scenario")

zone_rmse = np.sqrt(
    (11 / 17) * zone_mse["Без продаж"]
    + (6 / 17) * zone_mse["С продажами"]
)
display(zone_rmse.unstack("window").round(1))

window,M035–M051,M052–M068,M069–M085
zone,,,
1,73.2,38.2,44.5
2,63.7,35.1,43.3
3,82.7,68.9,69.4
4,139.3,78.5,148.7
5,141.1,99.6,113.2
6,311.5,75.6,273.7
7,145.3,96.2,206.2
8,73.0,48.2,49.0


### Выбранное решение и ограничения оценки

Используем сезонный уровень зоны, линейную поправку календаря и соперника
и половинную поправку собственных ранних продаж. В этой проверке
$S=126.736$ против $139.155$ у среднего по зоне: улучшение **8.92%**.
Общая ошибка меньше ориентира во всех трёх окнах.

Основная польза ранних продаж проявляется на 27-дневном срезе.
На семидневном срезе эффект слабее. Одна из причин — разные длительности
и неравномерность продаж: короткое наблюдение может плохо описывать весь итог.

Все три окна использовались для выбора параметров и метода, поэтому
ни одно из них не является независимым тестом. Эти числа характеризуют
качество при выборе модели. Истинный RMSE 136 будущих строк неизвестен.

## 9. Прогноз для 17 будущих матчей

На утро 15.12.2025 доступны итоги всех 85 обучающих матчей.
Для каждой будущей пары берём расписание и продажи строго до этого утра.
Порядок строк сохраняем непосредственно из шаблона.

In [13]:
test = template[keys].merge(calendar, on="match_id", how="left", sort=False)
test = test.merge(zones, on=keys, how="left", sort=False)
test["cutoff"] = forecast_date
test["has_sales"] = test["sales_open"] < forecast_date
test["sold"] = [
    sales_before(row.match_id, row.zone, forecast_date)
    for row in test.itertuples()
]

history = train.loc[train["date"] < forecast_date]
raw_prediction = forecast_tickets(history, test)
capacity = test["seats"] - test["season_tickets"]

submission = template.copy()
submission["tickets_total"] = apply_bounds(raw_prediction, test["sold"], capacity)

output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "predictions.csv"
submission.to_csv(output_path, index=False)

print(f"Сохранено {len(submission)} прогнозов: outputs/predictions.csv")
display(submission.head(8))

Сохранено 136 прогнозов: outputs/predictions.csv


,match_id,zone,tickets_total
0,M086,1,403
1,M086,2,216
2,M086,3,545
3,M086,4,805
4,M086,5,703
5,M086,6,543
6,M086,7,738
7,M086,8,381


## 10. Итог

Получен файл `outputs/predictions.csv`: 136 целых прогнозов полного числа
обычных билетов, по одному для каждой пары матч–зона.
Для матчей без продаж используется сезонная календарная модель;
для матчей с продажами добавляется поправка фактического темпа.
Результат ограничен уже проданным и доступными местами.

Решение целиком воспроизводится выполнением ячеек сверху вниз.
Для запуска нужны исходные четыре CSV, Python, pandas, numpy и scikit-learn.